# 2026 Bracket Sensitivity — Temperature & Upset Threshold

Explores how the bracket picks and champion probabilities change as we vary two post-training knobs:

| Knob | What it does | Default |
|------|-------------|------|
| `prob_temperature` | Softens all win probabilities toward 0.5 (T>1) or sharpens them (T<1). Applied via `sigmoid(logit(p)/T)`. | 1.0 |
| `upset_threshold` | Pick the lower seed (upset) when `P(higher wins) < threshold`. | 0.5 |

These two knobs are independent:
- **Temperature** changes the underlying probabilities (affects both MC advancement and greedy bracket picks).
- **Upset threshold** only changes which team is selected in the greedy bracket — it doesn't affect the simulated probabilities.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pickle
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

logging.basicConfig(level=logging.WARNING)  # suppress INFO during batch runs

from nba_predictor.config import cfg
from nba_predictor.models.ensemble import StackingEnsemble  # noqa: F401 — required for pickle
from nba_predictor.predict.bracket_simulator import (
    simulate_full_bracket,
    build_greedy_bracket,
    _load_model,
    _load_bracket_files,
    _fallback_log,
)

SEASON = 2026
model_dir = cfg.project_root / cfg.paths['models']['trained']

winner_model = _load_model(model_dir / 'ensemble_winner.pkl')
length_model_path = model_dir / 'lgbm_length.pkl'
length_model = _load_model(length_model_path) if length_model_path.exists() else None

bracket_input, team_store = _load_bracket_files(SEASON)
print(f'Loaded: {len(team_store)} teams, {len(bracket_input)} R1 matchups')

In [ ]:
# ── Parameter grid ────────────────────────────────────────────────────────────
# Temperatures: 1.0 (default), 1.25, 1.5, 2.0
# Thresholds:   0.50 (standard), 0.53, 0.55

TEMPERATURES  = [1.0, 1.25, 1.5, 2.0]
THRESHOLDS    = [0.50, 0.53, 0.55]

# Pre-compute one prob_table per temperature (expensive step, ~2s each)
# Each temperature produces a different set of 120 pairwise probabilities.
# For a given temperature, we apply every threshold to the same prob_table.

print('Pre-computing prob tables (one per temperature)...')
results_by_temp = {}
for T in TEMPERATURES:
    r = simulate_full_bracket(
        bracket_input, team_store, winner_model,
        season=SEASON, n_simulations=10_000,
        length_model=length_model, temperature=T,
    )
    results_by_temp[T] = r
    print(f'  T={T:.2f}  done  (fallback={len(_fallback_log)})')
print('Done.')

In [ ]:
# ── Build greedy brackets for every (T, threshold) combination ────────────────

brackets = {}  # (T, thresh) -> list[dict]
for T in TEMPERATURES:
    for thresh in THRESHOLDS:
        brackets[(T, thresh)] = build_greedy_bracket(
            results_by_temp[T], team_store,
            length_model=length_model, season=SEASON,
            upset_threshold=thresh,
        )

# Helper: extract ordered series for a bracket
ROUND_ORDER = ['first_round', 'second_round', 'conf_finals', 'nba_finals']
ROUND_SHORT  = {'first_round': 'R1', 'second_round': 'R2',
                'conf_finals': 'CF', 'nba_finals': 'Finals'}

def bracket_picks(bracket):
    """Return ordered list of (label, higher, lower, winner, p_higher) tuples."""
    rows = []
    for rnd in ROUND_ORDER:
        for conf in (['East','West'] if rnd != 'nba_finals' else ['Finals']):
            series = [s for s in bracket
                      if s['round'] == rnd and s.get('conference','Finals') == conf]
            for s in sorted(series, key=lambda x: x.get('higher_seed','')):
                label = f"{ROUND_SHORT[rnd]} {conf[:1]}: {s['higher_seed']}v{s['lower_seed']}"
                rows.append({
                    'label': label,
                    'higher': s['higher_seed'],
                    'lower':  s['lower_seed'],
                    'winner': s['predicted_winner'],
                    'p_higher': s['p_higher_seed_wins'],
                    'p_winner': s['p_winner'],
                    'length': s['modal_length'],
                    'round': rnd,
                    'conference': conf,
                })
    return rows

# Baseline picks (T=1.0, thresh=0.50)
baseline = bracket_picks(brackets[(1.0, 0.50)])
print(f'Bracket has {len(baseline)} series')

In [ ]:
# ── Cell 4: Side-by-side bracket comparison table ────────────────────────────
# Rows = matchups (15 series).  Columns = (T, threshold) settings.
# Green = favorite wins.  Orange = upset pick.  Bold = differs from baseline.

fig_width  = 3.5 + 2.5 * len(TEMPERATURES) * len(THRESHOLDS)
fig_height = 0.5 + 0.45 * len(baseline)
fig, ax = plt.subplots(figsize=(min(fig_width, 30), fig_height))
ax.axis('off')

cols = [(T, thr) for T in TEMPERATURES for thr in THRESHOLDS]
col_labels = [f'T={T:.2f}\ncut={thr:.2f}' for T, thr in cols]

row_labels  = [r['label'] for r in baseline]
n_rows = len(row_labels)
n_cols = len(cols)

# Colours
C_FAV    = '#d4edda'   # light green  — favourite wins
C_UPSET  = '#ffe5b4'   # light orange — upset pick
C_CHANGE = '#fff3cd'   # yellow highlight — differs from baseline
C_HEAD   = '#343a40'   # dark header

cell_text   = []
cell_colors = []
for row in baseline:
    txt_row  = []
    col_row  = []
    base_winner = row['winner']
    for T, thr in cols:
        b = brackets[(T, thr)]
        # Find matching series
        match = next(
            (s for s in b
             if s['round'] == row['round']
             and s['higher_seed'] == row['higher']
             and s['lower_seed']  == row['lower']),
            None
        )
        if match is None:
            txt_row.append('—')
            col_row.append('#ffffff')
            continue

        winner  = match['predicted_winner']
        p_h     = match['p_higher_seed_wins']
        is_upset = winner == match['lower_seed']
        differs  = winner != base_winner

        # Label: winner + probability from higher-seed's perspective
        txt_row.append(f"{winner}\n{p_h:.1%}")
        if differs:
            col_row.append(C_CHANGE)
        elif is_upset:
            col_row.append(C_UPSET)
        else:
            col_row.append(C_FAV)

    cell_text.append(txt_row)
    cell_colors.append(col_row)

table = ax.table(
    cellText=cell_text,
    rowLabels=row_labels,
    colLabels=col_labels,
    cellColours=cell_colors,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2.2)

# Style header row
for j in range(n_cols):
    table[0, j].set_facecolor(C_HEAD)
    table[0, j].set_text_props(color='white', fontweight='bold', fontsize=7)

# Style row labels
for i in range(n_rows):
    table[i+1, -1].set_facecolor('#f8f9fa')
    table[i+1, -1].set_text_props(fontsize=7, ha='right')

# Legend
legend_patches = [
    mpatches.Patch(facecolor=C_FAV,   label='Favourite wins'),
    mpatches.Patch(facecolor=C_UPSET, label='Upset pick'),
    mpatches.Patch(facecolor=C_CHANGE,label='Differs from baseline (T=1.0, cut=0.50)'),
]
ax.legend(handles=legend_patches, loc='lower right',
          bbox_to_anchor=(1.0, -0.02), fontsize=8, framealpha=0.9)

ax.set_title('2026 NBA Playoff — Bracket Picks by Temperature & Upset Threshold\n'
             '(p_higher shown; orange = upset; yellow = differs from baseline)',
             fontsize=11, pad=12)

plt.tight_layout()
plt.savefig('../reports/figures/bracket_sensitivity_table.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/figures/bracket_sensitivity_table.png')

In [ ]:
# ── Cell 5: Full bracket visual for 4 hand-picked scenarios ──────────────────
# Scenarios chosen to illustrate the range of settings.

SCENARIOS = [
    (1.00, 0.50, 'Baseline\n(T=1.0, cut=0.50)'),
    (1.00, 0.55, 'High threshold\n(T=1.0, cut=0.55)'),
    (1.50, 0.50, 'Softened probs\n(T=1.5, cut=0.50)'),
    (1.50, 0.55, 'Softened + threshold\n(T=1.5, cut=0.55)'),
]

# East / West structure — 15 series in 4 rounds
CONF_ROUND_SLOTS = {
    'East': {
        'first_round':  [(1,8),(2,7),(3,6),(4,5)],
        'second_round': [None, None],
        'conf_finals':  [None],
    },
    'West': {
        'first_round':  [(1,8),(2,7),(3,6),(4,5)],
        'second_round': [None, None],
        'conf_finals':  [None],
    },
}

def get_series(bracket, rnd, conf):
    """Sorted by higher_seed for consistent ordering."""
    return sorted(
        [s for s in bracket if s['round'] == rnd
         and s.get('conference', 'Finals') == conf],
        key=lambda s: int(team_store.loc[team_store.team == s['higher_seed'], 'seed'].values[0])
    )

def draw_bracket_panel(ax, bracket, title, team_store):
    """Draw one bracket panel into an Axes."""
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 22)
    ax.axis('off')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=4)

    # Colours
    FAV_BG   = '#e8f5e9'
    UPSET_BG = '#fff3e0'
    FAV_TXT  = '#1b5e20'
    UPSET_TXT= '#e65100'

    def box(ax, x, y, text, is_upset=False, prob=None):
        bg  = UPSET_BG if is_upset else FAV_BG
        fg  = UPSET_TXT if is_upset else FAV_TXT
        ax.add_patch(mpatches.FancyBboxPatch(
            (x, y - 0.35), 2.4, 0.7,
            boxstyle='round,pad=0.05',
            facecolor=bg, edgecolor='#aaaaaa', linewidth=0.5
        ))
        label = text if prob is None else f'{text}  {prob:.0%}'
        ax.text(x + 1.2, y, label, ha='center', va='center',
                fontsize=7, color=fg,
                fontweight='bold' if is_upset else 'normal')

    # Layout constants
    # Columns: R1-East(0), R2-East(1), CF-East(2), Finals(3), CF-West(4), R2-West(5), R1-West(6)
    # But we'll use a simpler top-down layout:
    # Row layout per conf:
    #   R1: 4 rows spaced at y = 18,15,12,9
    #   R2: 2 rows spaced at y = 16.5, 10.5
    #   CF: 1 row at y = 13.5

    conf_y_base = {'East': 0.5, 'West': 0.5}  # not used — kept for reference

    # East panel: x in [0, 2.6]
    # West panel: x in [3.0, 5.6]
    # Finals:     x in [6.0, 8.6]

    panels = [('East', 0.3), ('West', 3.0)]

    r1_ys = [20, 16.5, 13, 9.5]
    r2_ys = [18.25, 11.25]
    cf_y  = 14.75

    conf_champs = {}

    for conf, x_off in panels:
        r1 = get_series(bracket, 'first_round',  conf)
        r2 = get_series(bracket, 'second_round', conf)
        cf = get_series(bracket, 'conf_finals',  conf)

        # R1 header
        ax.text(x_off + 1.2, 21.2, f'{conf}', ha='center', va='bottom',
                fontsize=8, fontweight='bold', color='#333333')
        ax.text(x_off + 1.2, 20.7, 'Round 1', ha='center', va='bottom',
                fontsize=6, color='#777777')

        for i, s in enumerate(r1):
            is_up = s['predicted_winner'] == s['lower_seed']
            box(ax, x_off, r1_ys[i], s['predicted_winner'],
                is_upset=is_up, prob=s['p_higher_seed_wins'])

        # R2
        ax.text(x_off + 1.2, 20.7, '', ha='center', va='bottom', fontsize=6)
        for i, s in enumerate(r2):
            is_up = s['predicted_winner'] == s['lower_seed']
            box(ax, x_off, r2_ys[i], s['predicted_winner'],
                is_upset=is_up, prob=s['p_higher_seed_wins'])

        # CF
        for s in cf:
            is_up = s['predicted_winner'] == s['lower_seed']
            box(ax, x_off, cf_y, s['predicted_winner'],
                is_upset=is_up, prob=s['p_higher_seed_wins'])
            conf_champs[conf] = s['predicted_winner']

    # NBA Finals
    finals = [s for s in bracket if s['round'] == 'nba_finals']
    if finals:
        f = finals[0]
        is_up = f['predicted_winner'] == f['lower_seed']
        box(ax, 5.8, cf_y, f['predicted_winner'],
            is_upset=is_up, prob=f['p_higher_seed_wins'])
        ax.text(7.0, cf_y + 0.65, 'NBA FINALS', ha='center', va='bottom',
                fontsize=6.5, fontweight='bold', color='#b8860b')
        ax.text(7.0, cf_y - 0.65,
                f"{f['higher_seed']} vs {f['lower_seed']}",
                ha='center', va='top', fontsize=6, color='#555555')


fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, (T, thr, title) in zip(axes, SCENARIOS):
    draw_bracket_panel(ax, brackets[(T, thr)], title, team_store)

# Shared legend
legend_patches = [
    mpatches.Patch(facecolor='#e8f5e9', edgecolor='#aaaaaa', label='Favourite wins'),
    mpatches.Patch(facecolor='#fff3e0', edgecolor='#aaaaaa', label='Upset pick (bold orange)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))

fig.suptitle('2026 NBA Bracket Under Different Temperature & Threshold Settings\n'
             '(winner shown with P(higher seed wins) in parentheses)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('../reports/figures/bracket_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/figures/bracket_scenarios.png')

In [ ]:
# ── Cell 6: Champion probability heatmaps across the full (T, threshold) grid ─
# Top 8 teams — one subplot per team, colour = p_champion

# Collect champion probs for every (T, thresh) combination
champ_grid = {}  # team -> (T, thresh) -> p_champ

all_teams_ranked = sorted(
    results_by_temp[1.0]['champion_probs'],
    key=lambda t: -results_by_temp[1.0]['champion_probs'][t]
)
TOP_TEAMS = all_teams_ranked[:8]

for team in TOP_TEAMS:
    champ_grid[team] = {}
    for T in TEMPERATURES:
        for thr in THRESHOLDS:
            # champion_probs is MC-based, not affected by threshold (only MC sims)
            # but temperature does affect it
            p = results_by_temp[T]['champion_probs'].get(team, 0.0)
            champ_grid[team][(T, thr)] = p

# Build matrix: rows=temperature, cols=threshold
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

cmap = LinearSegmentedColormap.from_list('prob', ['#fff7f0','#fd8d3c','#d73027'])

for idx, team in enumerate(TOP_TEAMS):
    ax = axes[idx]
    matrix = np.array([[champ_grid[team][(T, thr)] for thr in THRESHOLDS]
                       for T in TEMPERATURES])
    im = ax.imshow(matrix, cmap=cmap, aspect='auto',
                   vmin=0, vmax=max(0.01, matrix.max() * 1.15))

    ax.set_xticks(range(len(THRESHOLDS)))
    ax.set_xticklabels([f'{t:.2f}' for t in THRESHOLDS], fontsize=8)
    ax.set_yticks(range(len(TEMPERATURES)))
    ax.set_yticklabels([f'{T:.2f}' for T in TEMPERATURES], fontsize=8)
    ax.set_xlabel('Upset threshold', fontsize=8)
    ax.set_ylabel('Temperature', fontsize=8)

    seed_val = int(team_store.loc[team_store.team == team, 'seed'].values[0])
    conf_val = team_store.loc[team_store.team == team, 'conference'].values[0]
    ax.set_title(f'{team}  ({conf_val[0]}{seed_val})', fontsize=10, fontweight='bold')

    for i in range(len(TEMPERATURES)):
        for j in range(len(THRESHOLDS)):
            v = matrix[i, j]
            ax.text(j, i, f'{v:.1%}', ha='center', va='center',
                    fontsize=9, color='black' if v < matrix.max()*0.7 else 'white',
                    fontweight='bold' if (i == 0 and j == 0) else 'normal')

    plt.colorbar(im, ax=ax, format='%.0%%', shrink=0.8)

fig.suptitle('Champion Probability by Team — Temperature × Upset Threshold Grid\n'
             '(MC-simulated; champion probs depend on temperature but not threshold)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/champion_prob_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/figures/champion_prob_heatmap.png')

In [ ]:
# ── Cell 7: First-round win probability ranges across temperature settings ─────
# Bar chart showing how each R1 matchup's P(higher wins) shifts with T.

r1_matchups = [
    (row['higher_seed'], row['lower_seed'])
    for _, row in bracket_input.iterrows()
]

fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(r1_matchups))
width = 0.18
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for i, T in enumerate(TEMPERATURES):
    prob_table = results_by_temp[T]['prob_table']
    probs = []
    for higher, lower in r1_matchups:
        p = prob_table.get((higher, lower), prob_table.get((lower, higher), 0.5))
        if (higher, lower) not in prob_table:
            p = 1 - p
        probs.append(p)
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, probs, width, label=f'T={T:.2f}', color=colors[i], alpha=0.85)

ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='50% (coin flip)')
ax.set_xticks(x)
ax.set_xticklabels(
    [f'{h} vs {l}' for h, l in r1_matchups],
    rotation=30, ha='right', fontsize=9
)
ax.set_ylabel('P(higher seed wins)', fontsize=10)
ax.set_title('First-Round Win Probabilities by Temperature\n'
             'Higher bar = stronger favourite; bars converge toward 0.5 as T increases',
             fontsize=11)
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/r1_probs_by_temperature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reports/figures/r1_probs_by_temperature.png')

In [ ]:
# ── Cell 8: Text summary — what changes from baseline ─────────────────────────

baseline_bracket = brackets[(1.0, 0.50)]
baseline_champ   = max(results_by_temp[1.0]['champion_probs'],
                       key=results_by_temp[1.0]['champion_probs'].get)

print('=' * 65)
print('BRACKET CHANGE SUMMARY vs BASELINE (T=1.0, threshold=0.50)')
print('=' * 65)

for T in TEMPERATURES:
    for thr in THRESHOLDS:
        if T == 1.0 and thr == 0.50:
            continue
        b   = brackets[(T, thr)]
        key = f'T={T:.2f}, cut={thr:.2f}'
        changes = []
        upset_picks = 0

        for s in b:
            if s['predicted_winner'] == s['lower_seed']:
                upset_picks += 1

            # Find matching baseline series
            base_s = next(
                (x for x in baseline_bracket
                 if x['round'] == s['round']
                 and x['higher_seed'] == s['higher_seed']
                 and x['lower_seed']  == s['lower_seed']),
                None
            )
            if base_s and base_s['predicted_winner'] != s['predicted_winner']:
                rnd = ROUND_SHORT.get(s['round'], s['round'])
                changes.append(
                    f"  {rnd} {s.get('conference','')[:1]}: "
                    f"{s['higher_seed']} vs {s['lower_seed']} → "
                    f"{base_s['predicted_winner']} ⟹ {s['predicted_winner']} "
                    f"(p={s['p_higher_seed_wins']:.1%})"
                )

        champ = max(results_by_temp[T]['champion_probs'],
                    key=results_by_temp[T]['champion_probs'].get)
        champ_p = results_by_temp[T]['champion_probs'][champ]

        print(f'\n── {key}  ({upset_picks} upset picks, champ: {champ} {champ_p:.1%}) ──')
        if changes:
            for c in changes:
                print(c)
        else:
            print('  (no changes from baseline)')